In [8]:
# ==============================================
# TTS - Computer Vision and Application
# Object Detection on Trash (TACO Dataset)
# Novelty: Image Enhancement using CLAHE
# Model: YOLOv8n
# ==============================================

!pip install ultralytics opencv-python matplotlib tqdm --quiet

import os
import cv2
import glob
import shutil
import random
import zipfile
import urllib.request
from tqdm import tqdm
import matplotlib.pyplot as plt
from ultralytics import YOLO

# ==============================================
# 1️⃣ Download Dataset (TACO)
# ==============================================
DATASET_URL = "https://github.com/pedropro/TACO/archive/refs/heads/master.zip"
DATA_DIR = "/content/datasets"
os.makedirs(DATA_DIR, exist_ok=True)

dataset_zip = os.path.join(DATA_DIR, "taco.zip")
print("📥 Downloading TACO dataset...")
urllib.request.urlretrieve(DATASET_URL, dataset_zip)

print("📦 Extracting...")
with zipfile.ZipFile(dataset_zip, 'r') as zip_ref:
    zip_ref.extractall(DATA_DIR)

# ==============================================
# 2️⃣ Detect Actual Image Folder Automatically
# ==============================================
print("🔍 Scanning for image folders...")
# Explicitly set the known correct image directory path
ORIG_IMG_DIR = "/content/datasets/TACO-master/data/restricted_unlimited"

if not os.path.exists(ORIG_IMG_DIR) or not glob.glob(f"{ORIG_IMG_DIR}/*.*g"):
    raise ValueError("⚠️ Tidak ditemukan folder berisi gambar di lokasi yang diharapkan!")

print(f"✅ Detected image folder: {ORIG_IMG_DIR}")

# ==============================================
# 3️⃣ Image Enhancement (CLAHE)
# ==============================================
ENH_IMG_DIR = os.path.join(DATA_DIR, "enhanced_images")
os.makedirs(ENH_IMG_DIR, exist_ok=True)

def apply_CLAHE(image):
    lab = cv2.cvtColor(image, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8,8))
    cl = clahe.apply(l)
    merged = cv2.merge((cl, a, b))
    return cv2.cvtColor(merged, cv2.COLOR_LAB2BGR)

print("✨ Enhancing images using CLAHE...")
for img_path in tqdm(glob.glob(f"{ORIG_IMG_DIR}/*.*g")):
    img = cv2.imread(img_path)
    if img is None:
        continue
    enh_img = apply_CLAHE(img)
    filename = os.path.basename(img_path)
    cv2.imwrite(os.path.join(ENH_IMG_DIR, filename), enh_img)

print("✅ Enhancement complete! Enhanced images saved to:", ENH_IMG_DIR)

# ==============================================
# 4️⃣ Visualize Sample Before vs After
# ==============================================
image_files = glob.glob(f"{ORIG_IMG_DIR}/*.*g")
sample_files = random.sample(image_files, min(3, len(image_files)))

for file in sample_files:
    ori = cv2.imread(file)
    enh = cv2.imread(os.path.join(ENH_IMG_DIR, os.path.basename(file)))

    plt.figure(figsize=(10,4))
    plt.subplot(1,2,1)
    plt.imshow(cv2.cvtColor(ori, cv2.COLOR_BGR2RGB))
    plt.title("Original")
    plt.axis('off')

    plt.subplot(1,2,2)
    plt.imshow(cv2.cvtColor(enh, cv2.COLOR_BGR2RGB))
    plt.title("Enhanced (CLAHE)")
    plt.axis('off')

    plt.show()

# ==============================================
# 5️⃣ Prepare a Small Subset (for demo)
# ==============================================
def prepare_subset(src_dir, dest_dir, n=500):
    os.makedirs(dest_dir, exist_ok=True)
    all_imgs = glob.glob(f"{src_dir}/*.*g")
    for img_path in tqdm(random.sample(all_imgs, min(n, len(all_imgs)))):
        shutil.copy(img_path, os.path.join(dest_dir, os.path.basename(img_path)))

ORI_TRAIN_DIR = "/content/ori_train"
ENH_TRAIN_DIR = "/content/enh_train"

print("📁 Preparing subsets (500 images each)...")
prepare_subset(ORIG_IMG_DIR, ORI_TRAIN_DIR)
prepare_subset(ENH_IMG_DIR, ENH_TRAIN_DIR)

# ==============================================
# 6️⃣ Train YOLOv8 on Original Dataset
# ==============================================
print("🚀 Training YOLOv8 on Original Images...")
model_ori = YOLO('yolov8n.pt')
model_ori.train(data='coco128.yaml', epochs=50, imgsz=640, project='/content/runs', name='ori_yolo')

# ==============================================
# 7️⃣ Train YOLOv8 on Enhanced Dataset
# ==============================================
print("🚀 Training YOLOv8 on Enhanced Images...")
model_enh = YOLO('yolov8n.pt')
model_enh.train(data='coco128.yaml', epochs=50, imgsz=640, project='/content/runs', name='enh_yolo')

# ==============================================
# 8️⃣ Evaluate & Compare
# ==============================================
print("📊 Evaluating both models...")
metrics_ori = model_ori.val()
metrics_enh = model_enh.val()

print("\n📈 COMPARISON RESULTS:")
print(f"Original Dataset   → mAP50: {metrics_ori.box.map50:.4f}, Precision: {metrics_ori.box.mp:.4f}, Recall: {metrics_ori.box.mr:.4f}")
print(f"Enhanced (CLAHE)   → mAP50: {metrics_enh.box.map50:.4f}, Precision: {metrics_enh.box.mp:.4f}, Recall: {metrics_enh.box.mr:.4f}")

# ==============================================
# 9️⃣ Visualize Detection Results
# ==============================================
sample_img = random.choice(glob.glob(f"{ORI_TRAIN_DIR}/*.*g"))

print("\n🔍 Predicting using Original-trained model...")
results_ori = model_ori.predict(source=sample_img, save=True)
print("\n🔍 Predicting using Enhanced-trained model...")
results_enh = model_enh.predict(source=sample_img, save=True)

print("\n✅ Detection results saved to /content/runs/detect/")

📥 Downloading TACO dataset...
📦 Extracting...
🔍 Scanning for image folders...


ValueError: ⚠️ Tidak ditemukan folder berisi gambar di lokasi yang diharapkan!